# 固定名义朝向：第一类与无乘子第二类拉格朗日方程
名义速度均为零，$\psi=\psi_0+\delta\psi$，$\theta=\delta\theta$。
不是把真实状态固定为目标，而是在该参考点保留能量的扰动二阶项，得到一阶动力学。
$x$ 保留现有数值仿真的**世界 x 位移扰动**；$\psi_0=0$ 时它就是名义前向位置。
$\cos\psi_0\ne0$ 是这个世界 x 速度参数化的适用条件。
此处不把偏航扰动强行设为0；保留小偏航控制通道。

In [1]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'base.py').exists():
    if root==root.parent: raise RuntimeError('找不到 theory/base.py')
    root=root.parent
sys.path.insert(0,str(root))
import sympy as s
import numpy as np
from IPython.display import display, Math
# 只改 LaTeX 显示，不替换计算符号：dx -> x 上点，ddx -> x 双点。
from base import q, qd, qdd
latex_names = {}
for coordinate, velocity, acceleration in zip(q, qd, qdd):
    name = s.latex(coordinate)
    latex_names[velocity] = r'\dot{' + name + '}'
    latex_names[acceleration] = r'\ddot{' + name + '}'
s.init_printing(symbol_names=latex_names)
np.set_printoptions(precision=8,suppress=True,linewidth=130)
def show(label, expression):
    display(Math(label+' = '+s.latex(expression, symbol_names=latex_names)))


In [2]:
from straight_balance.lagrange import *
from lagrange_utils import tau
m=build()

## 从共享运动学到本场景能量

In [3]:
show('T',m['T']);show('V',m['V'])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

保留精确表达式供对账；上面 T、V 是后续实际求导的二阶近似。

In [4]:
show('T_{exact}',m['T_exact']);show('V_{exact}',m['V_exact'])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 第一类：六个广义方向与三个约束反力
坐标顺序 $q=[x,y,\psi,\theta,\theta_L,\theta_R]^T$。

$$M_6\ddot q+h_6=B_6\tau+A_c^T\lambda,\quad A_c\dot q=0,$$
$$\begin{bmatrix}M_6&-A_c^T\\ A_c&0\end{bmatrix}
\begin{bmatrix}\ddot q\\\lambda\end{bmatrix}=
\begin{bmatrix}B_6\tau-h_6\\-\dot A_c\dot q\end{bmatrix}.$$

电机反作用力矩同时作用于车体：$Q_\theta=-(\tau_L+\tau_R)$，不能漏掉。
下面列出全部分块矩阵、偏置和最终9×9系数矩阵；$\lambda$ 的数值依赖约束行的归一化。

In [5]:
for key in ('M6','h6','B6','Acon','augmented_matrix','augmented_rhs'):show(key.replace('_',r'\_'),m[key])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 第二类：消除乘子，保留非完整约束的修正项
选取可允许独立速度 $u$，使 $\dot q=S(q)u$、$A_c S=0$。
令 $p_q=\partial L/\partial\dot q$（这里的 $p_q$ 是动量，不是俯仰角速度 p）。

$$\frac{d}{dt}(S^Tp_q)-S^T\frac{\partial L}{\partial q}-\dot S^Tp_q=S^TB_6\tau.$$

这就是无乘子的拉格朗日—达朗贝尔方程，本文称“第二类形式”。它**不等于**先把速度约束代入 L 再直接对三个假想全局坐标套普通 Euler–Lagrange 算子。

$$M=S^TM_6S,\quad h=S^T(M_6\dot S u+h_6),\quad B_\tau=S^TB_6,$$
$$M\dot u+h=B_\tau\tau.$$

代码还从上面的动量公式独立构造 M_second、h_second，并与投影式逐项比较。

In [6]:
show('S',m['S']);show('u',m['u'])
for key in ('M','h','B'):show(key,m[key])
assert clean(m['M']-m['M_second'])==s.zeros(3)
assert clean(m['h']-m['h_second'])==s.zeros(3,1)
print('PASS：两种拉格朗日形式严格一致')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

PASS：两种拉格朗日形式严格一致


## 最终局部矩阵
$$M_0\delta\ddot z+C_0\delta\dot z+K_0\delta z=B_0\tau,\quad z=[x,\psi,\theta]^T.$$
一阶局部模型的约束也冻结在同一静止参考点；不把局部能量模型假装成完整非线性模型。

In [7]:
for key,label in [('M','M_0'),('C','C_0'),('K','K_0'),('B','B_0')]:show(label,m[key])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>